# 读取数据

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np

# 1) 读 CSV
df = pd.read_csv("../data/7.匹配时间/merged_output.csv")

# 2) 读城市边界 GeoJSON
city_gdf = gpd.read_file("/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_市.geojson")

city_gdf

In [ ]:
# 计算中心点，并分别提取经度（x）和纬度（y）作为新字段
city_gdf['center_lon'] = city_gdf.geometry.centroid.x  # 经度
city_gdf['center_lat'] = city_gdf.geometry.centroid.y  # 纬度

city_gdf

# 转换经纬度

In [ ]:
# 3) 只保留需要字段，修一下几何（可选但推荐）
city_gdf = city_gdf[["name", "geometry"]].copy()
city_gdf["geometry"] = city_gdf["geometry"].buffer(0)  # 修复少量无效几何（常见且有效）

# 4) CRS 对齐（你的 geojson 通常是经纬度 WGS84，但有时 crs 为空）
#    点数据一定是 EPSG:4326（经纬度）
if city_gdf.crs is None:
    city_gdf = city_gdf.set_crs(epsg=4326)
else:
    city_gdf = city_gdf.to_crs(epsg=4326)

# 5) 把 lng/lat 转成数值，生成点 GeoDataFrame
df["lng"] = pd.to_numeric(df["lng"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")

points_gdf = gpd.GeoDataFrame(
    df[["lng", "lat"]].copy(),
    geometry=gpd.points_from_xy(df["lng"], df["lat"]),
    crs="EPSG:4326")


# 空间连接到天地图城市

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# =========================
# 参数区：按需修改
# =========================
LNG_COL = "lng"
LAT_COL = "lat"
CITY_NAME_COL = "name"          # city_gdf 里城市名称字段
OUTPUT_COL = "city_name"        # 写回 df 的列名

PREDICATE = "within"            # "within" 或 "intersects"
USE_NEAREST_FALLBACK = True     # 是否对 NaN 做 nearest 兜底
MAX_NEAREST_DIST_M = None       # e.g. 50000 表示 50km，None 表示不限制

# =========================
# 0) 基础清洗：经纬度有效性
# =========================
df = df.copy()

# 转数值，非法变 NaN
df[LNG_COL] = pd.to_numeric(df[LNG_COL], errors="coerce")
df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")

# 可选：过滤明显不合法的经纬度
valid_coord = (
    df[LNG_COL].between(-180, 180, inclusive="both") &
    df[LAT_COL].between(-90, 90, inclusive="both")
)

# =========================
# 1) 构建点 GeoDataFrame（WGS84）
# =========================
points_gdf = gpd.GeoDataFrame(
    df.loc[valid_coord].copy(),
    geometry=gpd.points_from_xy(df.loc[valid_coord, LNG_COL], df.loc[valid_coord, LAT_COL]),
    crs="EPSG:4326"
)

# =========================
# 2) 确保 city_gdf 有 CRS，并与点一致（用于 within/intersects）
# =========================
if city_gdf.crs is None:
    raise ValueError("city_gdf.crs is None：请先给城市面数据设置正确的 CRS（例如 EPSG:4326）。")

# 先把城市面转换到 EPSG:4326，以便和 points_gdf 做空间连接
city_4326 = city_gdf.to_crs(points_gdf.crs)

# =========================
# 3) 空间连接：点落在哪个城市面
# =========================
joined = gpd.sjoin(
    points_gdf,
    city_4326[[CITY_NAME_COL, "geometry"]],
    how="left",
    predicate=PREDICATE
)

# 处理：极少数一个点命中多个面 -> 取第一个
city_name = joined[CITY_NAME_COL].groupby(joined.index).first()

# 先创建输出列
df[OUTPUT_COL] = pd.NA
# 回填到原 df（对齐索引）
df.loc[city_name.index, OUTPUT_COL] = city_name

# =========================
# 4) nearest 兜底：只对仍是 NaN 的点做
# =========================
if USE_NEAREST_FALLBACK:
    mask_na = df[OUTPUT_COL].isna() & valid_coord

    if mask_na.any():
        # 只取需要兜底的点
        points_na = gpd.GeoDataFrame(
            df.loc[mask_na].copy(),
            geometry=gpd.points_from_xy(df.loc[mask_na, LNG_COL], df.loc[mask_na, LAT_COL]),
            crs="EPSG:4326"
        )

        # --- 关键修复：投影到米制 CRS 再 nearest（避免 warning & 距离更准确） ---
        # 自动估算一个适合此区域的 UTM CRS（单位：米）
        proj_crs = city_4326.estimate_utm_crs()

        points_proj = points_na.to_crs(proj_crs)
        city_proj = city_4326.to_crs(proj_crs)

        nearest = gpd.sjoin_nearest(
            points_proj,
            city_proj[[CITY_NAME_COL, "geometry"]],
            how="left",
            distance_col="nearest_dist_m"
        )

        # 如果你想限制最大兜底距离（比如海上点别硬匹配）
        if MAX_NEAREST_DIST_M is not None:
            nearest.loc[nearest["nearest_dist_m"] > MAX_NEAREST_DIST_M, CITY_NAME_COL] = pd.NA

        # --- 关键修复：按 index 对齐回填，避免 .values 顺序错位 ---
        df.loc[nearest.index, OUTPUT_COL] = nearest[CITY_NAME_COL]

# 进行统计

In [ ]:
# 筛选列 进行保存
cols = ["vehicle","city_name"]
df_cols = df[cols].copy()


# vehicle 可能是 0.0/1.0 这种浮点，先转成整数（并去掉缺失）
df_cols = df_cols.dropna(subset=["vehicle", "city_name"])
df_cols["vehicle"] = df_cols["vehicle"].astype(int)


# 统计 city_name x vehicle 的计数
city_vehicle_counts = (df_cols.groupby(["city_name", "vehicle"])
                         .size()
                         .unstack(fill_value=0))
# 保证 0~4 都在列上（即使某城市没出现某类也显示为 0）
city_vehicle_counts = city_vehicle_counts.reindex(columns=[0, 1, 2, 3], fill_value=0)


# city_vehicle_counts 的列是 [0,1,2,3,4] 统计所有的类别总数
city_vehicle_counts["sum"] = city_vehicle_counts[[0, 1, 2, 3]].sum(axis=1)


# 整理列名
city_vehicle_counts.columns.name = None
# 把 index（city_name）变成普通列 -> 表头一行
city_vehicle_counts = city_vehicle_counts.reset_index()



# 计算每个类型的占比
cols = [0, 1, 2, 3]

# 防止 sum=0 导致除零（如果你确定没有0，可以删掉这行）
denom = city_vehicle_counts['sum'].replace(0, np.nan)
city_vehicle_counts[[f'{c}_ratio' for c in cols]] = city_vehicle_counts[cols].div(denom, axis=0)


In [ ]:
city_vehicle_counts

# 空间连接，每个省市的中心点 city_vehicle_counts 和 city_gdf

In [ ]:
# 为了节省内存并保持结果整洁，我们只从 city_gdf 取出需要的列
columns_to_merge = city_gdf[['name', 'center_lon', 'center_lat']]

# 使用 merge 进行连接
# left_on='city_name' 表示左表(city_vehicle_counts)用来连接的列名
# right_on='name' 表示右表(columns_to_merge)用来连接的列名
# how='left' 表示以 city_vehicle_counts 为基础进行左连接，保留其所有行
result_df = pd.merge(
    city_vehicle_counts, 
    columns_to_merge, 
    left_on='city_name', 
    right_on='name', 
    how='left'
)

# 连接后会同时存在 'city_name' 和 'name' 两列内容相同的数据，可以将 'name' 列删除
city_vehicle_counts = result_df.drop(columns=['name'])

In [ ]:
city_vehicle_counts

# 绘制堆叠柱状图 

In [ ]:
import matplotlib.pyplot as plt

# 设置支持中文的字体（例如 SimHei），同时确保负号能正常显示
plt.rcParams['font.sans-serif'] = ['SimHei']
# plt.rcParams['font.sans-serif'] = ['Times New Roman', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 横着绘制

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== 可自由调节 =====
top_n = 10
metric_col = "0_ratio"   # 可改成 0 / 1 / 2 / 3 / "sum"
count_cols = [0, 1, 2, 3]  # 需要堆叠的类别列
city_col = "city_name"
# ======================

# 兼容列名是 int 或 str 的情况
def _col(x):
    return x if x in city_vehicle_counts.columns else str(x)

metric_col = _col(metric_col)
count_cols = [_col(c) for c in count_cols]
city_col = _col(city_col)

# 取Top N城市
top_cities = (
    city_vehicle_counts
    .sort_values(metric_col, ascending=False)
    .head(top_n)
    .reset_index(drop=True))

# 计算百分比（100% 堆叠）
row_total = top_cities[count_cols].sum(axis=1).replace(0, np.nan)
pct = top_cities[count_cols].div(row_total, axis=0) * 100

# 画图（水平柱状图，城市名在左侧一列）
y = np.arange(len(top_cities)) # 将 x 换成 y 变量名更符合语境
cmap = plt.get_cmap("Spectral")
colors = [cmap(v) for v in np.linspace(0.1, 0.9, len(count_cols))]

# 调整画布大小：宽度可以固定，高度随城市数量自适应
fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.45))) 

# 将 bottom 替换为 left，记录横向堆叠的起始位置
left = np.zeros(len(top_cities)) 

for i, c in enumerate(count_cols):
    # 核心修改 1：使用 barh 绘制水平柱状，将 y 放在第一个参数，用 left 替代 bottom
    ax.barh(y, pct[c].fillna(0).values, left=left, color=colors[i], label=str(c))
    left += pct[c].fillna(0).values

# 核心修改 2：把所有 X 轴和 Y 轴的设置对调
ax.set_yticks(y)
# 城市名放在Y轴，通常水平排版即可，不需要 rotation=90
ax.set_yticklabels(top_cities[city_col].astype(str), ha="right") 
ax.set_xlim(0, 100) # 原先的 ylim 改为 xlim
ax.set_xlabel("Proportion（%）") # 原先的 ylabel 改为 xlabel
ax.set_title(f"Top {top_n} city of the car accidents")

# 体验优化：默认 barh 从下往上画，翻转 Y 轴让第一名在最上面
ax.invert_yaxis()

# 如果需要显示图例，把这行注释打开
# ax.legend(title="类别", ncol=min(4, len(count_cols)), bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

## 竖着绘制

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== 可自由调节 =====
top_n = 10
metric_col = "0_ratio"   # 可改成 0 / 1 / 2 / 3 / "sum"
count_cols = [0, 1, 2, 3]  # 需要堆叠的类别列
city_col = "city_name"
# ======================

# 兼容列名是 int 或 str 的情况
def _col(x):
    return x if x in city_vehicle_counts.columns else str(x)

metric_col = _col(metric_col)
count_cols = [_col(c) for c in count_cols]
city_col = _col(city_col)

# 取Top N城市
top_cities = (
    city_vehicle_counts
    .sort_values(metric_col, ascending=False)
    .head(top_n)
    .reset_index(drop=True))

# 计算百分比（100% 堆叠）
row_total = top_cities[count_cols].sum(axis=1).replace(0, np.nan)
pct = top_cities[count_cols].div(row_total, axis=0) * 100

# 画图（竖直柱状，城市名在底部一行）
x = np.arange(len(top_cities))
cmap = plt.get_cmap("Spectral")
colors = [cmap(v) for v in np.linspace(0.1, 0.9, len(count_cols))]

fig, ax = plt.subplots(figsize=(max(12, top_n * 0.45), 7))
bottom = np.zeros(len(top_cities))

for i, c in enumerate(count_cols):
    ax.bar(x, pct[c].fillna(0).values, bottom=bottom, color=colors[i], label=str(c))
    bottom += pct[c].fillna(0).values

ax.set_xticks(x)
ax.set_xticklabels(top_cities[city_col].astype(str), rotation=90, ha="center")
ax.set_ylim(0, 100)
ax.set_ylabel("占比（%）")
ax.set_title(f"Top {top_n} 城市 100%堆叠柱状图（按 {metric_col} 排序）")
# ax.legend(title="类别", ncol=min(4, len(count_cols)), bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()


# 绘制对应城市的分布（拿到城市）

In [ ]:
city_vehicle_counts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== 可自由调节 =====
top_n = 30
metric_col = "0_ratio"   # 可改成 0 / 1 / 2 / 3 / "sum" 按哪一个排序？
count_cols = [0, 1, 2, 3]  # 需要堆叠的类别列
city_col = "city_name"
# ======================

# 兼容列名是 int 或 str 的情况
def _col(x):
    return x if x in city_vehicle_counts.columns else str(x)

metric_col = _col(metric_col)
count_cols = [_col(c) for c in count_cols]
city_col = _col(city_col)

# 取Top N城市
top_cities = (
    city_vehicle_counts
    .sort_values(metric_col, ascending=False)
    .head(top_n)
    .reset_index(drop=True))

# 绘制地图

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as cx

# 1) DataFrame -> GeoDataFrame (经纬度默认是WGS84: EPSG:4326)
gdf = gpd.GeoDataFrame(
    top_cities.copy(),
    geometry=gpd.points_from_xy(top_cities["center_lon"], top_cities["center_lat"]),
    crs="EPSG:4326")

# 2) 转到 Web Mercator（contextily底图切片用）
gdf_3857 = gdf.to_crs(epsg=3857)

# 3) 画图
fig, ax = plt.subplots(figsize=(6, 4),dpi=300)

# 红色坐标点
gdf_3857.plot(
    ax=ax,
    color="blue",
    markersize=25,
    alpha=0.85,
    zorder=3
)

# 4) 加底图（按你要求的写法）
plot_crs = gdf_3857.crs
cx.add_basemap(
    ax,
    source=cx.providers.CartoDB.Positron,
    crs=plot_crs
)

# 5) 缩放到点的范围（加一点边距）
minx, miny, maxx, maxy = gdf_3857.total_bounds
padx = (maxx - minx) * 0.2 if maxx > minx else 100000
pady = (maxy - miny) * 0.2 if maxy > miny else 100000
ax.set_xlim(minx - padx, maxx + padx)
ax.set_ylim(miny - pady, maxy + pady)

ax.set_axis_off()
ax.set_title("城市经纬度分布（中国）", fontsize=14)
plt.tight_layout()
plt.show()
